# Protein Folding QUBO: Complete Analysis Suite

This notebook combines three progressive protein folding examples, demonstrating the QUBO formulation from small tractable problems to larger, more realistic scenarios.

## Overview

1. **Part 1: 8-Residue Tractable Example** (2×4 lattice, 64 variables)
   - Small enough to enumerate all Hamiltonian paths
   - Verifies QUBO correctness against brute-force ground truth
   - Demonstrates Simulated Annealing effectiveness

2. **Part 2: 24-Residue Problem** (6×4 lattice, 576 variables)
   - Intermediate scale
   - Tests SA on more complex conformational space

3. **Part 3: 32-Residue Problem** (8×4 lattice, 1,024 variables)
   - Larger illustrative 2D problem
   - Ground truth from exhaustive DFS enumeration (77,968 paths)
   - Optimal E_MJ = -14.25

## Key Concepts

### Binary Variable Encoding

For an N-residue chain on an M-position lattice:
- **Binary variable**: `b[i,n]` where:
  - `i` = residue index (0 to N-1)
  - `n` = lattice position index (0 to M-1)
  - `b[i,n] = 1` if residue i is at position n, else 0

- **Total variables**: N × M

### Enumeration Strategy: Hamiltonian Paths

A valid protein conformation is a **Hamiltonian path** on the lattice graph:
- Visits each lattice position **exactly once**
- Consecutive residues occupy **adjacent positions** (connected by an edge)
- This naturally encodes:
  - Chain connectivity (residues i and i+1 must be neighbors)
  - No overlaps (each position used once)
  - One residue per position

**Enumeration approach**:
1. Build adjacency matrix of the lattice
2. Use depth-first search (DFS) to find all Hamiltonian paths
3. For each path, compute contact energy using the supplied signed matrix
4. Find minimum energy (ground truth)

### Energy Function

**E_total = E_MJ + λ₁·E₁ + λ₂·E₂ + λ₃·E₃**

Where:
- **E_MJ**: Contact energy from the supplied signed matrix
- **E₁**: One position per residue constraint
- **E₂**: No two residues at same position constraint
- **E₃**: Chain connectivity constraint

### Contact energy and constraints

For each unordered pair of residues separated by at least two chain links, the contact term adds the supplied signed coefficient when their lattice sites are nearest neighbors. Each residue pair is counted once. Negative coefficients lower the energy.

The supplied 20×20 matrix does not match Table VI of Miyazawa and Jernigan (1985), which the cited 2025 paper identifies as its source. The exact energies reported below are therefore exact for this repository's illustrative 2D contact model, not reproductions of the paper's 3D benchmarks.

The other QUBO terms enforce one site per residue (E₁), no overlap (E₂), and chain connectivity (E₃). E₃ penalizes consecutive residues on distinct non-neighboring sites; E₂ penalizes same-site placements.


---
---

# PART 1: 8-Residue Tractable Example

---
---


---
# 8. Protein Folding with Simulated Annealing: A Tractable Example

This section demonstrates the protein folding QUBO on a **small, tractable problem** where we can:

1. **Enumerate all valid conformations** via brute force
2. **Find the true global optimum**
3. **Verify that Simulated Annealing finds it**

We use:
- **8-residue chain** on a **2×4 lattice** (8 positions)
- **64 binary variables** (8 residues × 8 positions)
- **Supplied 20-type signed contact matrix; not the cited MJ Table VI; provenance is unknown**

This is small enough to enumerate all Hamiltonian paths (~thousands) but large enough to be interesting.

## 8.1 Setup

In [ ]:
from importlib.util import find_spec
missing = [name for name in ('numpy', 'matplotlib', 'pandas', 'dimod', 'neal')
           if find_spec(name) is None]
if missing:
    raise ImportError(f"Missing {missing}. Install the repository requirements.txt before running this notebook.")


In [ ]:
import numpy as np
import math
import matplotlib.pyplot as plt
from collections import defaultdict
from itertools import permutations
import time

from dimod import BinaryQuadraticModel
from neal import SimulatedAnnealingSampler

print("✓ Imports successful")

## 8.2 Supplied Contact Matrix

The supplied 20×20 signed contact matrix (different from the cited MJ Table VI).

In [ ]:
# One-letter amino acid codes
AMINO_ACIDS = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I',
               'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

# Miyazawa-Jernigan contact energies
# Negative values = favorable (attractive) interactions
MJ_MATRIX = np.array([
    [-0.20, -0.34, -0.44, -0.42, -0.62, -0.40, -0.42, -0.29, -0.51, -0.73, -0.65, -0.36, -0.64, -0.69, -0.31, -0.34, -0.40, -0.61, -0.53, -0.68],
    [-0.34, -0.15, -0.44, -0.90, -0.55, -0.50, -0.93, -0.40, -0.33, -0.42, -0.45, -0.35, -0.34, -0.41, -0.24, -0.37, -0.32, -0.26, -0.29, -0.37],
    [-0.44, -0.44, -0.28, -0.46, -0.58, -0.44, -0.51, -0.46, -0.54, -0.51, -0.47, -0.46, -0.41, -0.47, -0.34, -0.48, -0.50, -0.36, -0.42, -0.46],
    [-0.42, -0.90, -0.46, -0.19, -0.52, -0.53, -0.49, -0.44, -0.49, -0.42, -0.35, -0.62, -0.33, -0.36, -0.31, -0.48, -0.44, -0.23, -0.30, -0.36],
    [-0.62, -0.55, -0.58, -0.52, -1.54, -0.65, -0.55, -0.54, -0.73, -0.92, -0.88, -0.47, -0.92, -1.01, -0.49, -0.56, -0.61, -0.84, -0.79, -0.91],
    [-0.40, -0.50, -0.44, -0.53, -0.65, -0.29, -0.55, -0.40, -0.49, -0.56, -0.51, -0.46, -0.47, -0.54, -0.31, -0.43, -0.41, -0.39, -0.42, -0.51],
    [-0.42, -0.93, -0.51, -0.49, -0.55, -0.55, -0.22, -0.43, -0.42, -0.44, -0.37, -0.64, -0.35, -0.40, -0.30, -0.47, -0.43, -0.21, -0.27, -0.38],
    [-0.29, -0.40, -0.46, -0.44, -0.54, -0.40, -0.43, -0.14, -0.46, -0.52, -0.46, -0.41, -0.47, -0.51, -0.26, -0.37, -0.39, -0.44, -0.44, -0.47],
    [-0.51, -0.33, -0.54, -0.49, -0.73, -0.49, -0.42, -0.46, -0.34, -0.66, -0.62, -0.35, -0.64, -0.72, -0.37, -0.46, -0.47, -0.62, -0.57, -0.60],
    [-0.73, -0.42, -0.51, -0.42, -0.92, -0.56, -0.44, -0.52, -0.66, -1.00, -0.95, -0.41, -0.96, -1.04, -0.49, -0.52, -0.59, -0.90, -0.83, -0.98],
    [-0.65, -0.45, -0.47, -0.35, -0.88, -0.51, -0.37, -0.46, -0.62, -0.95, -0.91, -0.37, -0.93, -1.01, -0.46, -0.47, -0.53, -0.89, -0.80, -0.93],
    [-0.36, -0.35, -0.46, -0.62, -0.47, -0.46, -0.64, -0.41, -0.35, -0.41, -0.37, -0.18, -0.32, -0.39, -0.23, -0.39, -0.36, -0.19, -0.24, -0.35],
    [-0.64, -0.34, -0.41, -0.33, -0.92, -0.47, -0.35, -0.47, -0.64, -0.96, -0.93, -0.32, -0.89, -0.98, -0.45, -0.47, -0.52, -0.86, -0.78, -0.92],
    [-0.69, -0.41, -0.47, -0.36, -1.01, -0.54, -0.40, -0.51, -0.72, -1.04, -1.01, -0.39, -0.98, -1.10, -0.50, -0.51, -0.57, -0.98, -0.89, -1.02],
    [-0.31, -0.24, -0.34, -0.31, -0.49, -0.31, -0.30, -0.26, -0.37, -0.49, -0.46, -0.23, -0.45, -0.50, -0.16, -0.30, -0.32, -0.42, -0.38, -0.45],
    [-0.34, -0.37, -0.48, -0.48, -0.56, -0.43, -0.47, -0.37, -0.46, -0.52, -0.47, -0.39, -0.47, -0.51, -0.30, -0.33, -0.41, -0.41, -0.41, -0.48],
    [-0.40, -0.32, -0.50, -0.44, -0.61, -0.41, -0.43, -0.39, -0.47, -0.59, -0.53, -0.36, -0.52, -0.57, -0.32, -0.41, -0.38, -0.45, -0.45, -0.55],
    [-0.61, -0.26, -0.36, -0.23, -0.84, -0.39, -0.21, -0.44, -0.62, -0.90, -0.89, -0.19, -0.86, -0.98, -0.42, -0.41, -0.45, -0.86, -0.76, -0.84],
    [-0.53, -0.29, -0.42, -0.30, -0.79, -0.42, -0.27, -0.44, -0.57, -0.83, -0.80, -0.24, -0.78, -0.89, -0.38, -0.41, -0.45, -0.76, -0.69, -0.78],
    [-0.68, -0.37, -0.46, -0.36, -0.91, -0.51, -0.38, -0.47, -0.60, -0.98, -0.93, -0.35, -0.92, -1.02, -0.45, -0.48, -0.55, -0.84, -0.78, -0.96],
])

print(f"MJ Matrix: {MJ_MATRIX.shape}")
print(f"Energy range: [{MJ_MATRIX.min():.2f}, {MJ_MATRIX.max():.2f}]")
print(f"\nStrongest attractions (most negative):")
print(f"  C-C: {MJ_MATRIX[AA_TO_IDX['C'], AA_TO_IDX['C']]:.2f}")
print(f"  F-F: {MJ_MATRIX[AA_TO_IDX['F'], AA_TO_IDX['F']]:.2f}")
print(f"  I-F: {MJ_MATRIX[AA_TO_IDX['I'], AA_TO_IDX['F']]:.2f}")

## 8.3 The 2×4 Lattice

A small 2D lattice with 8 positions:
```
0 - 1 - 2 - 3
|   |   |   |
4 - 5 - 6 - 7
```

In [ ]:
def build_2d_lattice(rows, cols):
    """Build adjacency matrix for 2D rectangular lattice."""
    n = rows * cols
    adj = np.zeros((n, n), dtype=int)

    for i in range(n):
        r, c = i // cols, i % cols
        # Right neighbor
        if c < cols - 1:
            adj[i, i+1] = adj[i+1, i] = 1
        # Down neighbor
        if r < rows - 1:
            adj[i, i+cols] = adj[i+cols, i] = 1

    return adj

# 2x4 lattice
ROWS, COLS = 2, 4
NUM_POSITIONS = ROWS * COLS
adj_matrix = build_2d_lattice(ROWS, COLS)

print(f"Lattice: {ROWS}×{COLS} = {NUM_POSITIONS} positions")
print(f"\nAdjacency matrix:")
print(adj_matrix)

# Visualize
print(f"\nLattice layout:")
for r in range(ROWS):
    row_str = " - ".join([str(r*COLS + c) for c in range(COLS)])
    print(f"  {row_str}")
    if r < ROWS - 1:
        print("  " + "   ".join(["|"] * COLS))

## 8.4 Test Sequence

We'll use an 8-residue sequence with a mix of amino acid types to create interesting interactions.

In [ ]:
# Design a sequence with strong potential contacts
# Put hydrophobic residues (C, F, I, L, V, W, Y, M) that attract each other
# and some polar/charged (K, D, R, E) that may repel

SEQUENCE = "CFIKDLWV"  # 8 residues

print(f"Sequence: {SEQUENCE}")
print(f"Length: {len(SEQUENCE)}")
print(f"\nResidue types:")
for i, aa in enumerate(SEQUENCE):
    idx = AA_TO_IDX[aa]
    print(f"  {i}: {aa} (index {idx})")

print(f"\nPotential non-consecutive contacts (|i-j| > 1):")
for i in range(len(SEQUENCE)):
    for j in range(i+2, len(SEQUENCE)):
        e = MJ_MATRIX[AA_TO_IDX[SEQUENCE[i]], AA_TO_IDX[SEQUENCE[j]]]
        print(f"  {SEQUENCE[i]}({i}) - {SEQUENCE[j]}({j}): {e:.2f}")

## 8.5 Brute Force: Enumerate All Valid Conformations

A valid conformation is a **Hamiltonian path** on the lattice - visiting all 8 positions exactly once with consecutive residues at adjacent positions.

We enumerate all such paths and compute the MJ energy for each.

In [ ]:
def find_all_hamiltonian_paths(adj, n):
    """
    Find all Hamiltonian paths in a graph.
    Returns list of paths, where each path is a list of node indices.
    """
    paths = []

    def dfs(current, visited, path):
        if len(path) == n:
            paths.append(path.copy())
            return

        for next_node in range(n):
            if next_node not in visited and adj[current, next_node] == 1:
                visited.add(next_node)
                path.append(next_node)
                dfs(next_node, visited, path)
                path.pop()
                visited.remove(next_node)

    # Try starting from each node
    for start in range(n):
        dfs(start, {start}, [start])

    return paths


def compute_mj_energy(path, sequence, adj):
    """
    Compute MJ energy for a given path (conformation).

    E_MJ = -sum_{|i-j|>1} C(ai,aj) * delta(path[i] adjacent to path[j])
    """
    n = len(sequence)
    energy = 0.0

    for i in range(n):
        for j in range(i + 2, n):  # Non-consecutive residues
            pos_i, pos_j = path[i], path[j]
            if adj[pos_i, pos_j] == 1:  # Adjacent on lattice
                C_ij = MJ_MATRIX[AA_TO_IDX[sequence[i]], AA_TO_IDX[sequence[j]]]
                energy += C_ij  # Signed contact energy: negative is favorable

    return energy


# Find all Hamiltonian paths
print("Finding all Hamiltonian paths...")
start_time = time.time()
all_paths = find_all_hamiltonian_paths(adj_matrix, NUM_POSITIONS)
enum_time = time.time() - start_time

print(f"Found {len(all_paths)} valid conformations in {enum_time:.2f}s")

# Compute energy for each
print("\nComputing energies...")
path_energies = []
for path in all_paths:
    e = compute_mj_energy(path, SEQUENCE, adj_matrix)
    path_energies.append((e, path))

# Sort by energy
path_energies.sort(key=lambda x: x[0])

print(f"\nEnergy distribution:")
energies = [e for e, p in path_energies]
print(f"  Min: {min(energies):.4f}")
print(f"  Max: {max(energies):.4f}")
print(f"  Mean: {np.mean(energies):.4f}")
print(f"  Std: {np.std(energies):.4f}")

In [ ]:
# Show the best conformations
print("="*60)
print("TOP 10 CONFORMATIONS (Ground Truth)")
print("="*60)

def visualize_path(path, sequence, rows, cols):
    """Create a visual representation of a path on the lattice."""
    grid = [['.' for _ in range(cols)] for _ in range(rows)]
    for i, pos in enumerate(path):
        r, c = pos // cols, pos % cols
        grid[r][c] = sequence[i]
    return grid

OPTIMAL_ENERGY = path_energies[0][0]
OPTIMAL_PATH = path_energies[0][1]

print(f"\n*** OPTIMAL ENERGY: {OPTIMAL_ENERGY:.4f} ***\n")

for rank, (energy, path) in enumerate(path_energies[:10]):
    print(f"Rank {rank+1}: E = {energy:.4f}")
    print(f"  Path: {path}")
    grid = visualize_path(path, SEQUENCE, ROWS, COLS)
    for row in grid:
        print(f"    {' '.join(row)}")
    print()

In [ ]:
# Plot energy distribution
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.hist(energies, bins=30, edgecolor='black', alpha=0.7)
plt.axvline(OPTIMAL_ENERGY, color='red', linestyle='--', linewidth=2, label=f'Optimal: {OPTIMAL_ENERGY:.2f}')
plt.xlabel('E_MJ')
plt.ylabel('Count')
plt.title('Energy Distribution of All Valid Conformations')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, len(energies)+1), sorted(energies), 'b-', linewidth=0.5)
plt.axhline(OPTIMAL_ENERGY, color='red', linestyle='--', linewidth=2)
plt.xlabel('Rank')
plt.ylabel('E_MJ')
plt.title('Sorted Energies')

plt.tight_layout()
plt.show()

# Count how many solutions are near optimal
near_optimal = sum(1 for e in energies if e <= OPTIMAL_ENERGY + 0.1)
print(f"\nConformations within 0.1 of optimal: {near_optimal}")

## 8.6 QUBO Construction

Now we build the QUBO and test SA against our known ground truth.

In [ ]:
def build_protein_qubo(sequence, adj, lambda1=1.5, lambda2=2.0, lambda3=2.0):
    """
    Build QUBO for protein folding.

    E = E_MJ + λ1*E1 + λ2*E2 + λ3*E3
    """
    N = len(sequence)
    M = adj.shape[0]

    linear = defaultdict(float)
    quadratic = defaultdict(float)
    offset = 0.0

    def bit_idx(residue, position):
        return residue * M + position

    # E_MJ: Interaction energy
    for i in range(N):
        for j in range(i + 2, N):
            C_ij = MJ_MATRIX[AA_TO_IDX[sequence[i]], AA_TO_IDX[sequence[j]]]
            for n in range(M):
                for m in range(M):
                    if adj[n, m] == 1:
                        b_i = bit_idx(i, n)
                        b_j = bit_idx(j, m)
                        key = (min(b_i, b_j), max(b_i, b_j))
                        quadratic[key] += C_ij  # Matrix already contains signed energies

    # E1: One position per residue
    for i in range(N):
        for n in range(M):
            linear[bit_idx(i, n)] += lambda1 * (-1)
        for n in range(M):
            for m in range(n + 1, M):
                key = (bit_idx(i, n), bit_idx(i, m))
                quadratic[key] += lambda1 * 2
        offset += lambda1

    # E2: One residue per position
    for n in range(M):
        for i in range(N):
            for j in range(i + 1, N):
                key = (bit_idx(i, n), bit_idx(j, n))
                quadratic[key] += lambda2

    # E3: Chain connectivity
    non_adj = 1 - adj - np.eye(M)
    for i in range(N - 1):
        for n in range(M):
            for m in range(M):
                if non_adj[n, m] == 1:
                    b_i = bit_idx(i, n)
                    b_j = bit_idx(i + 1, m)
                    key = (min(b_i, b_j), max(b_i, b_j))
                    quadratic[key] += lambda3

    return dict(linear), dict(quadratic), offset


# Build QUBO with stronger penalties for this small problem
LAMBDA1, LAMBDA2, LAMBDA3 = 3.0, 4.0, 4.0  # Stronger than paper's (1.5, 2, 2)

linear, quadratic, offset = build_protein_qubo(
    SEQUENCE, adj_matrix,
    lambda1=LAMBDA1, lambda2=LAMBDA2, lambda3=LAMBDA3
)

num_bits = len(SEQUENCE) * NUM_POSITIONS
print(f"QUBO Statistics:")
print(f"  Variables: {num_bits}")
print(f"  Linear terms: {len(linear)}")
print(f"  Quadratic terms: {len(quadratic)}")
print(f"  Lagrange multipliers: λ=({LAMBDA1}, {LAMBDA2}, {LAMBDA3})")

## 8.7 Simulated Annealing

In [ ]:
def validate_solution(sample, sequence, adj):
    """Validate solution and compute E_MJ."""
    N = len(sequence)
    M = adj.shape[0]

    if set(sample) != set(range(N * M)) or any(value not in (0, 1) for value in sample.values()):
        raise ValueError('sample must contain exactly N*M binary variables')

    # Build bit matrix
    b = np.zeros((N, M))
    for bit_idx, val in sample.items():
        if val == 1:
            i, n = bit_idx // M, bit_idx % M
            if i < N:
                b[i, n] = 1

    # Check constraints
    E1 = np.sum((np.sum(b, axis=1) - 1) ** 2)
    E2 = sum(np.sum(b[:, n]) * (np.sum(b[:, n]) - 1) / 2 for n in range(M))

    non_adj = 1 - adj - np.eye(M)
    E3 = sum(b[i, :] @ non_adj @ b[i+1, :] for i in range(N-1))

    # Compute E_MJ
    E_MJ = 0
    for i in range(N):
        for j in range(i + 2, N):
            C_ij = MJ_MATRIX[AA_TO_IDX[sequence[i]], AA_TO_IDX[sequence[j]]]
            for n in range(M):
                for m in range(M):
                    if adj[n, m] == 1:
                        E_MJ += C_ij * b[i, n] * b[j, m]

    is_valid = (E1 == 0) and (E2 == 0) and (E3 == 0)

    # Extract path if valid
    path = None
    if is_valid:
        path = [None] * N
        for i in range(N):
            for n in range(M):
                if b[i, n] == 1:
                    path[i] = n

    return is_valid, E_MJ, {'E1': E1, 'E2': E2, 'E3': E3}, path

print("✓ Validation function defined")

In [ ]:
# Run SA
print("="*60)
print("SIMULATED ANNEALING")
print(f"Target (from brute force): E_MJ = {OPTIMAL_ENERGY:.4f}")
print("="*60)

bqm = BinaryQuadraticModel(linear, quadratic, offset, vartype='BINARY')
sampler = SimulatedAnnealingSampler()

# Test with different parameters
configs = [
    {'num_reads': 50, 'num_sweeps': 1000},
    {'num_reads': 50, 'num_sweeps': 5000},
    {'num_reads': 100, 'num_sweeps': 10000},
]

for config in configs:
    print(f"\n--- {config['num_sweeps']} sweeps, {config['num_reads']} reads ---")

    start = time.time()
    result = sampler.sample(bqm, **config, beta_range=(1.0, 50.0), seed=42)
    elapsed = time.time() - start

    # Analyze results
    valid_solutions = []
    for datum in result.data():
        is_valid, e_mj, violations, path = validate_solution(datum.sample, SEQUENCE, adj_matrix)
        direct_total = e_mj + LAMBDA1*violations['E1'] + LAMBDA2*violations['E2'] + LAMBDA3*violations['E3']
        if not np.isclose(datum.energy, direct_total, rtol=0, atol=1e-6):
            raise AssertionError('Sampler QUBO energy disagrees with direct evaluation')
        if is_valid:
            valid_solutions.append((e_mj, path, datum.energy))

    print(f"  Time: {elapsed:.2f}s")
    print(f"  Valid solutions: {len(valid_solutions)}/{config['num_reads']}")

    if valid_solutions:
        valid_solutions.sort(key=lambda x: x[0])
        best_e_mj, best_path, best_qubo = valid_solutions[0]
        print(f"  Best E_MJ found: {best_e_mj:.4f}")
        print(f"  Optimal E_MJ:    {OPTIMAL_ENERGY:.4f}")
        print(f"  Gap: {best_e_mj - OPTIMAL_ENERGY:.4f}")

        if abs(best_e_mj - OPTIMAL_ENERGY) < 0.001:
            print(f"  ✓ FOUND OPTIMAL SOLUTION!")
            print(f"  Path: {best_path}")
    else:
        # Show best invalid solution
        best = result.first
        _, _, violations, _ = validate_solution(best.sample, SEQUENCE, adj_matrix)
        print(f"  No valid solutions. Best violations: E1={violations['E1']:.0f}, E2={violations['E2']:.0f}, E3={violations['E3']:.0f}")

In [ ]:
# More detailed run
print("\n" + "="*60)
print("DETAILED SA ANALYSIS")
print("="*60)

result = sampler.sample(
    bqm,
    num_reads=200,
    num_sweeps=10000,
    beta_range=(1.0, 100.0),
    seed=123
)

# Collect all valid solutions
valid_results = []
for datum in result.data():
    is_valid, e_mj, violations, path = validate_solution(datum.sample, SEQUENCE, adj_matrix)
    direct_total = e_mj + LAMBDA1*violations['E1'] + LAMBDA2*violations['E2'] + LAMBDA3*violations['E3']
    if not np.isclose(datum.energy, direct_total, rtol=0, atol=1e-6):
        raise AssertionError('Sampler QUBO energy disagrees with direct evaluation')
    if is_valid:
        valid_results.append({'E_MJ': e_mj, 'path': path, 'QUBO': datum.energy})

print(f"\nValid solutions found: {len(valid_results)}/200")

if valid_results:
    # Sort by E_MJ
    valid_results.sort(key=lambda x: x['E_MJ'])

    print(f"\nBest SA solution:")
    best = valid_results[0]
    print(f"  E_MJ: {best['E_MJ']:.4f}")
    print(f"  Path: {best['path']}")

    print(f"\nGround truth (brute force):")
    print(f"  E_MJ: {OPTIMAL_ENERGY:.4f}")
    print(f"  Path: {OPTIMAL_PATH}")

    # Check if SA found optimal
    found_optimal = any(abs(r['E_MJ'] - OPTIMAL_ENERGY) < 0.001 for r in valid_results)
    if found_optimal:
        n_optimal = sum(1 for r in valid_results if abs(r['E_MJ'] - OPTIMAL_ENERGY) < 0.001)
        print(f"\n✓ SA found the optimal solution {n_optimal} times!")
    else:
        print(f"\n✗ SA did not find optimal. Gap: {best['E_MJ'] - OPTIMAL_ENERGY:.4f}")

    # Distribution of SA energies vs all energies
    sa_energies = [r['E_MJ'] for r in valid_results]

    plt.figure(figsize=(10, 4))
    plt.hist(energies, bins=30, alpha=0.5, label='All conformations', density=True)
    plt.hist(sa_energies, bins=15, alpha=0.7, label='SA solutions', density=True)
    plt.axvline(OPTIMAL_ENERGY, color='red', linestyle='--', linewidth=2, label=f'Optimal: {OPTIMAL_ENERGY:.2f}')
    plt.xlabel('E_MJ')
    plt.ylabel('Density')
    plt.title('SA Solutions vs All Valid Conformations')
    plt.legend()
    plt.show()

part1_results = {
    'label': '8-residue', 'variables': 64, 'paths': len(all_paths),
    'exact': OPTIMAL_ENERGY, 'valid_rate': len(valid_results) / 200 * 100,
    'best': min((r['E_MJ'] for r in valid_results), default=None),
    'reads': 200, 'sweeps': 10000, 'seed': 123,
}


## 8.8 Conclusions

This small example demonstrates:

1. **Ground truth verification**: We enumerated all valid conformations to find the true optimal
2. **SA effectiveness**: Simulated Annealing can find optimal/near-optimal solutions
3. **QUBO correctness**: The encoding correctly captures the protein folding constraints

### Scaling Notes

| Problem Size | Variables | Valid Conformations | Brute Force Time |
|--------------|-----------|---------------------|------------------|
| 8 residues, 2×4 lattice | 64 | 28 | < 1 second |
| 12 residues, 3×4 lattice | 144 | ~thousands | seconds |
| 48 residues, 4×4×3 lattice | 2,304 | ~10^11 | impractical |

For larger problems:
- **SA** works but needs more sweeps and careful tuning
- The cited paper reports hybrid annealing times for its distinct 48-residue, 3D benchmark.
- **QAOA** cannot scale beyond ~25-30 qubits for classical simulation


---
---

# PART 2: 24-Residue Problem

---
---


# Protein Folding QUBO: 24-Residue Scaling Test

- **24-residue chain** on a **4×6 lattice** (24 positions)
- **576 binary variables** (24 × 24)
- **Supplied 20-type signed contact matrix; not the cited MJ Table VI; provenance is unknown**

**Energy convention**: More negative E_total = better (lower energy)

**Terminology**:
- **Valid**: All constraints satisfied (E₁ = E₂ = E₃ = 0)
- **Optimal**: Valid solution with lowest (most negative) E_total

In [ ]:
from importlib.util import find_spec
missing = [name for name in ('numpy', 'matplotlib', 'pandas', 'dimod', 'neal')
           if find_spec(name) is None]
if missing:
    raise ImportError(f"Missing {missing}. Install the repository requirements.txt before running this notebook.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import time
import pandas as pd

from dimod import BinaryQuadraticModel
from neal import SimulatedAnnealingSampler

np.random.seed(42)
print("✓ Imports successful")

## Supplied Contact Matrix

**Convention**: Negative values are favorable (unlike the H/P/C toy matrix). These coefficients do not match the cited Table VI.

In [ ]:
AMINO_ACIDS = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I',
               'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

# MJ potential: negative = favorable (opposite of H/P/C convention)
MJ_MATRIX = np.array([
    [-0.20, -0.34, -0.44, -0.42, -0.62, -0.40, -0.42, -0.29, -0.51, -0.73, -0.65, -0.36, -0.64, -0.69, -0.31, -0.34, -0.40, -0.61, -0.53, -0.68],
    [-0.34, -0.15, -0.44, -0.90, -0.55, -0.50, -0.93, -0.40, -0.33, -0.42, -0.45, -0.35, -0.34, -0.41, -0.24, -0.37, -0.32, -0.26, -0.29, -0.37],
    [-0.44, -0.44, -0.28, -0.46, -0.58, -0.44, -0.51, -0.46, -0.54, -0.51, -0.47, -0.46, -0.41, -0.47, -0.34, -0.48, -0.50, -0.36, -0.42, -0.46],
    [-0.42, -0.90, -0.46, -0.19, -0.52, -0.53, -0.49, -0.44, -0.49, -0.42, -0.35, -0.62, -0.33, -0.36, -0.31, -0.48, -0.44, -0.23, -0.30, -0.36],
    [-0.62, -0.55, -0.58, -0.52, -1.54, -0.65, -0.55, -0.54, -0.73, -0.92, -0.88, -0.47, -0.92, -1.01, -0.49, -0.56, -0.61, -0.84, -0.79, -0.91],
    [-0.40, -0.50, -0.44, -0.53, -0.65, -0.29, -0.55, -0.40, -0.49, -0.56, -0.51, -0.46, -0.47, -0.54, -0.31, -0.43, -0.41, -0.39, -0.42, -0.51],
    [-0.42, -0.93, -0.51, -0.49, -0.55, -0.55, -0.22, -0.43, -0.42, -0.44, -0.37, -0.64, -0.35, -0.40, -0.30, -0.47, -0.43, -0.21, -0.27, -0.38],
    [-0.29, -0.40, -0.46, -0.44, -0.54, -0.40, -0.43, -0.14, -0.46, -0.52, -0.46, -0.41, -0.47, -0.51, -0.26, -0.37, -0.39, -0.44, -0.44, -0.47],
    [-0.51, -0.33, -0.54, -0.49, -0.73, -0.49, -0.42, -0.46, -0.34, -0.66, -0.62, -0.35, -0.64, -0.72, -0.37, -0.46, -0.47, -0.62, -0.57, -0.60],
    [-0.73, -0.42, -0.51, -0.42, -0.92, -0.56, -0.44, -0.52, -0.66, -1.00, -0.95, -0.41, -0.96, -1.04, -0.49, -0.52, -0.59, -0.90, -0.83, -0.98],
    [-0.65, -0.45, -0.47, -0.35, -0.88, -0.51, -0.37, -0.46, -0.62, -0.95, -0.91, -0.37, -0.93, -1.01, -0.46, -0.47, -0.53, -0.89, -0.80, -0.93],
    [-0.36, -0.35, -0.46, -0.62, -0.47, -0.46, -0.64, -0.41, -0.35, -0.41, -0.37, -0.18, -0.32, -0.39, -0.23, -0.39, -0.36, -0.19, -0.24, -0.35],
    [-0.64, -0.34, -0.41, -0.33, -0.92, -0.47, -0.35, -0.47, -0.64, -0.96, -0.93, -0.32, -0.89, -0.98, -0.45, -0.47, -0.52, -0.86, -0.78, -0.92],
    [-0.69, -0.41, -0.47, -0.36, -1.01, -0.54, -0.40, -0.51, -0.72, -1.04, -1.01, -0.39, -0.98, -1.10, -0.50, -0.51, -0.57, -0.98, -0.89, -1.02],
    [-0.31, -0.24, -0.34, -0.31, -0.49, -0.31, -0.30, -0.26, -0.37, -0.49, -0.46, -0.23, -0.45, -0.50, -0.16, -0.30, -0.32, -0.42, -0.38, -0.45],
    [-0.34, -0.37, -0.48, -0.48, -0.56, -0.43, -0.47, -0.37, -0.46, -0.52, -0.47, -0.39, -0.47, -0.51, -0.30, -0.33, -0.41, -0.41, -0.41, -0.48],
    [-0.40, -0.32, -0.50, -0.44, -0.61, -0.41, -0.43, -0.39, -0.47, -0.59, -0.53, -0.36, -0.52, -0.57, -0.32, -0.41, -0.38, -0.45, -0.45, -0.55],
    [-0.61, -0.26, -0.36, -0.23, -0.84, -0.39, -0.21, -0.44, -0.62, -0.90, -0.89, -0.19, -0.86, -0.98, -0.42, -0.41, -0.45, -0.86, -0.76, -0.84],
    [-0.53, -0.29, -0.42, -0.30, -0.79, -0.42, -0.27, -0.44, -0.57, -0.83, -0.80, -0.24, -0.78, -0.89, -0.38, -0.41, -0.45, -0.76, -0.69, -0.78],
    [-0.68, -0.37, -0.46, -0.36, -0.91, -0.51, -0.38, -0.47, -0.60, -0.98, -0.93, -0.35, -0.92, -1.02, -0.45, -0.48, -0.55, -0.84, -0.78, -0.96],
])

print(f"MJ Matrix: {MJ_MATRIX.shape}")
print(f"Energy range: [{MJ_MATRIX.min():.2f}, {MJ_MATRIX.max():.2f}]")

## Lattice and Sequence Setup

In [ ]:
def build_2d_lattice(rows, cols):
    n = rows * cols
    adj = np.zeros((n, n), dtype=int)
    for i in range(n):
        r, c = i // cols, i % cols
        if c < cols - 1:
            adj[i, i+1] = adj[i+1, i] = 1
        if r < rows - 1:
            adj[i, i+cols] = adj[i+cols, i] = 1
    return adj

ROWS, COLS = 4, 6
NUM_POSITIONS = ROWS * COLS
adj_matrix = build_2d_lattice(ROWS, COLS)

# First 24 residues of Sequence 5 from Irbäck et al.
FULL_SEQ_5 = "ENDVTMDMDPSPCLFRIHNLPRAHSFDRFGWHQFDKYHYKWKWAWAPS"
SEQUENCE = FULL_SEQ_5[:24]

print(f"Lattice: {ROWS}×{COLS} = {NUM_POSITIONS} positions")
print(f"Sequence: {SEQUENCE}")
print(f"Length: {len(SEQUENCE)}")
print(f"Variables: {len(SEQUENCE) * NUM_POSITIONS}")

## Energy Calculation Functions

Based on `calc_mods.py`, adapted for MJ matrix convention.

**Key difference**: MJ values are already negative for favorable contacts, so we do NOT negate in `compute_E_MJ`.

**E_total = E_MJ + λ₁·E₁ + λ₂·E₂ + λ₃·E₃**

In [ ]:
# Energy calculation functions (from calc_mods.py, adapted for MJ matrix)

def compute_E_MJ(chain, b, adj, verbose=False):
    """
    Compute interaction energy.

    For MJ matrix: values are already negative for favorable contacts,
    so we return E directly (no negation, unlike H/P/C version).
    """
    N = len(chain)
    M = adj.shape[0]
    E = 0.0
    contacts = []

    for i in range(N):
        for j in range(i+2, N):  # |i-j| > 1 means j >= i+2
            a_i, a_j = chain[i], chain[j]
            c_ij = MJ_MATRIX[AA_TO_IDX[a_i], AA_TO_IDX[a_j]]
            for n in range(M):
                for m in range(M):
                    if b[i, n] == 1 and b[j, m] == 1 and adj[n, m] == 1:
                        E += c_ij
                        contacts.append((i, j, a_i, a_j, n, m, c_ij))
                        if verbose:
                            print(f"  Contact: residues {i}({a_i}) and {j}({a_j}) at sites {n} and {m}, C={c_ij}")

    if verbose:
        print(f"  Total E_MJ: {E}")

    return E  # No negation for MJ matrix


def compute_E1(b):
    """
    Constraint: Each residue occupies exactly one site.
    E1 = sum_i (sum_n b_{i,n} - 1)^2
    """
    result = np.sum((np.sum(b, axis=1) - 1) ** 2)
    return int(result)


def compute_E2(b):
    """
    Constraint: Each site has at most one residue.
    E2 = (1/2) * sum_n sum_{i!=j} b_{i,n} b_{j,n}
    """
    M = b.shape[1]
    pair_count = 0
    for n in range(M):
        site_occupancy = np.sum(b[:, n])
        pair_count += site_occupancy * (site_occupancy - 1)
    return int(0.5 * pair_count)


def compute_E3(b, adj):
    """
    Constraint: Chain connectivity - consecutive residues must be adjacent.
    """
    N = b.shape[0]
    M = adj.shape[0]
    non_adj = 1 - adj - np.eye(M)

    E3 = 0.0
    for i in range(N - 1):
        E3 += b[i, :] @ non_adj @ b[i+1, :]

    return int(E3)


def total_energy(chain, b, adj, L1=1.0, L2=1.0, L3=1.0, verbose=False):
    """
    Compute total QUBO energy: E_total = E_MJ + L1*E1 + L2*E2 + L3*E3
    """
    E_MJ = compute_E_MJ(chain, b, adj, verbose)
    E_1 = compute_E1(b)
    E_2 = compute_E2(b)
    E_3 = compute_E3(b, adj)

    total = E_MJ + L1*E_1 + L2*E_2 + L3*E_3

    if verbose:
        print(f"\nEnergy components:")
        print(f"  E_MJ = {E_MJ:.4f}")
        print(f"  E_1  = {E_1} (position uniqueness)")
        print(f"  E_2  = {E_2} (self-avoidance)")
        print(f"  E_3  = {E_3} (connectivity)")
        print(f"  E_total = {total:.4f}")

    return total, (E_MJ, E_1, E_2, E_3)


def is_valid(breakdown):
    """Check if configuration satisfies all constraints (E1=E2=E3=0)"""
    _, E1, E2, E3 = breakdown
    return E1 == 0 and E2 == 0 and E3 == 0


print("✓ Energy functions defined (based on calc_mods.py)")

## QUBO Construction

In [ ]:
def build_protein_qubo(sequence, adj, lambda1=1.5, lambda2=2.0, lambda3=2.0):
    """
    Build QUBO for protein folding.
    Minimizing QUBO minimizes E_total (most negative = best).
    """
    N = len(sequence)
    M = adj.shape[0]

    linear = defaultdict(float)
    quadratic = defaultdict(float)
    offset = 0.0

    def bit_idx(residue, position):
        return residue * M + position

    # E_MJ: Add C_ij directly (MJ values already negative for favorable)
    for i in range(N):
        for j in range(i + 2, N):
            C_ij = MJ_MATRIX[AA_TO_IDX[sequence[i]], AA_TO_IDX[sequence[j]]]
            for n in range(M):
                for m in range(M):
                    if adj[n, m] == 1:
                        b_i, b_j = bit_idx(i, n), bit_idx(j, m)
                        key = (min(b_i, b_j), max(b_i, b_j))
                        quadratic[key] += C_ij

    # E1: One position per residue - (sum_n b_{i,n} - 1)^2
    for i in range(N):
        for n in range(M):
            linear[bit_idx(i, n)] += lambda1 * (-1)
        for n in range(M):
            for m in range(n + 1, M):
                key = (bit_idx(i, n), bit_idx(i, m))
                quadratic[key] += lambda1 * 2
        offset += lambda1

    # E2: One residue per position
    for n in range(M):
        for i in range(N):
            for j in range(i + 1, N):
                key = (bit_idx(i, n), bit_idx(j, n))
                quadratic[key] += lambda2

    # E3: Chain connectivity
    non_adj = 1 - adj - np.eye(M)
    for i in range(N - 1):
        for n in range(M):
            for m in range(M):
                if non_adj[n, m] == 1:
                    b_i, b_j = bit_idx(i, n), bit_idx(i + 1, m)
                    key = (min(b_i, b_j), max(b_i, b_j))
                    quadratic[key] += lambda3

    return dict(linear), dict(quadratic), offset


# Build QUBOs with different λ configurations
lambda_configs = [(1.5, 2.0, 2.0), (2.0, 3.0, 3.0), (3.0, 4.0, 4.0)]

print("Building QUBOs...")
qubos = {}
for lam in lambda_configs:
    linear, quadratic, offset = build_protein_qubo(SEQUENCE, adj_matrix, *lam)
    qubos[lam] = (linear, quadratic, offset)
    print(f"  λ={lam}: {len(quadratic)} quadratic terms")

## Solution Validation

In [ ]:
def validate_solution(sample, sequence, adj, L1, L2, L3):
    """
    Validate a solution and compute E_total.
    Returns: (is_valid, E_total, breakdown)
    """
    N = len(sequence)
    M = adj.shape[0]

    if set(sample) != set(range(N * M)) or any(value not in (0, 1) for value in sample.values()):
        raise ValueError('sample must contain exactly N*M binary variables')

    # Build bit matrix from sample
    b = np.zeros((N, M))
    for bit_idx, val in sample.items():
        if val == 1:
            i, n = bit_idx // M, bit_idx % M
            if i < N:
                b[i, n] = 1

    E_total, breakdown = total_energy(sequence, b, adj, L1, L2, L3)
    valid = is_valid(breakdown)

    return valid, E_total, breakdown


print("✓ Validation function defined")

## Brute-Force Enumeration (Reference)

In [ ]:
def find_hamiltonian_paths_limited(adj, n, max_paths=10000, time_limit=120):
    """Find Hamiltonian paths with limits."""
    paths = []
    start_time = time.time()
    completed = True

    def dfs(current, visited, path):
        nonlocal completed
        if len(paths) >= max_paths or time.time() - start_time > time_limit:
            completed = False
            return False
        if len(path) == n:
            paths.append(path.copy())
            return True
        for next_node in range(n):
            if next_node not in visited and adj[current, next_node] == 1:
                visited.add(next_node)
                path.append(next_node)
                if not dfs(next_node, visited, path):
                    path.pop()
                    visited.remove(next_node)
                    return False
                path.pop()
                visited.remove(next_node)
        return True

    for start in range(n):
        if len(paths) >= max_paths or time.time() - start_time > time_limit:
            completed = False
            break
        dfs(start, {start}, [start])

    return paths, completed


def compute_path_energy(path, sequence, adj):
    """Compute E_MJ for a given path (valid conformation)."""
    N = len(sequence)
    E = 0.0
    for i in range(N):
        for j in range(i + 2, N):
            if adj[path[i], path[j]] == 1:
                E += MJ_MATRIX[AA_TO_IDX[sequence[i]], AA_TO_IDX[sequence[j]]]
    return E  # This IS E_total for valid paths (since E1=E2=E3=0)


print("Enumerating Hamiltonian paths...")
start_time = time.time()
all_paths, enumeration_complete = find_hamiltonian_paths_limited(
    adj_matrix, NUM_POSITIONS, max_paths=50000, time_limit=180
)
enum_time = time.time() - start_time

print(f"Found {len(all_paths)} paths in {enum_time:.1f}s")
print(f"Enumeration complete: {enumeration_complete}")

if len(all_paths) > 0:
    print("\nComputing energies...")
    path_energies = [(compute_path_energy(p, SEQUENCE, adj_matrix), p) for p in all_paths]
    path_energies.sort(key=lambda x: x[0])  # Sort ascending (most negative first = best)

    BEST_ENUMERATED_ENERGY = path_energies[0][0]
    OPTIMAL_ENERGY = BEST_ENUMERATED_ENERGY if enumeration_complete else None
    energies = [e for e, p in path_energies]

    print(f"\nEnergy statistics (for valid conformations):")
    print(f"  Best (most negative):  {min(energies):.4f}")
    print(f"  Worst (least negative): {max(energies):.4f}")
    print(f"  Mean: {np.mean(energies):.4f}")

    if enumeration_complete:
        print(f"\n*** VERIFIED OPTIMAL: E_total = {OPTIMAL_ENERGY:.4f} ***")
    else:
        print(f"\n*** BEST FOUND (incomplete): E_total = {BEST_ENUMERATED_ENERGY:.4f} ***")
else:
    OPTIMAL_ENERGY = None
    print("No paths found.")

## Simulated Annealing Experiments

In [ ]:
def run_sa_experiment(linear, quadratic, offset, num_reads, num_sweeps, beta_range,
                      sequence, adj, L1, L2, L3, seed=None):
    """Run SA and return statistics."""
    bqm = BinaryQuadraticModel(linear, quadratic, offset, vartype='BINARY')
    sampler = SimulatedAnnealingSampler()

    start = time.time()
    result = sampler.sample(bqm, num_reads=num_reads, num_sweeps=num_sweeps,
                            beta_range=beta_range, seed=seed)
    solve_time = time.time() - start

    valid_solutions = []
    all_results = []

    for datum in result.data():
        valid, E_total, breakdown = validate_solution(datum.sample, sequence, adj, L1, L2, L3)
        if not np.isclose(datum.energy, E_total, rtol=0, atol=1e-6):
            raise AssertionError('Sampler QUBO energy disagrees with direct evaluation')
        all_results.append({'valid': valid, 'E_total': E_total,
                            'QUBO_energy': datum.energy, 'breakdown': breakdown})
        if valid:
            valid_solutions.append({'E_total': E_total, 'QUBO_energy': datum.energy})

    return {
        'solve_time': solve_time,
        'num_reads': num_reads,
        'num_sweeps': num_sweeps,
        'valid_count': len(valid_solutions),
        'valid_solutions': valid_solutions,
        'valid_rate': len(valid_solutions) / num_reads * 100,
        'all_results': all_results
    }

In [ ]:
print("="*70)
print("SIMULATED ANNEALING EXPERIMENTS")
print("="*70)
if OPTIMAL_ENERGY is not None:
    print(f"Reference optimal E_total = {OPTIMAL_ENERGY:.4f}")
print()

all_exp_results = []

test_configs = [
    ((1.5, 2.0, 2.0), 5000, 100, (1.0, 50.0)),
    ((1.5, 2.0, 2.0), 10000, 100, (1.0, 50.0)),
    ((2.0, 3.0, 3.0), 5000, 100, (1.0, 50.0)),
    ((2.0, 3.0, 3.0), 10000, 100, (1.0, 50.0)),
    ((3.0, 4.0, 4.0), 5000, 100, (1.0, 50.0)),
    ((3.0, 4.0, 4.0), 10000, 100, (1.0, 50.0)),
    ((3.0, 4.0, 4.0), 20000, 100, (1.0, 100.0)),
]

for lam, sweeps, reads, beta in test_configs:
    print(f"λ={lam}, sweeps={sweeps}")

    if lam not in qubos:
        qubos[lam] = build_protein_qubo(SEQUENCE, adj_matrix, *lam)
    linear, quadratic, offset = qubos[lam]

    result = run_sa_experiment(linear, quadratic, offset, reads, sweeps, beta,
                               SEQUENCE, adj_matrix, lam[0], lam[1], lam[2], seed=42)
    result['lambda'] = lam
    all_exp_results.append(result)

    print(f"  Time: {result['solve_time']:.2f}s")
    print(f"  Valid: {result['valid_count']}/{reads} ({result['valid_rate']:.1f}%)")
    if result['valid_solutions']:
        best_e = min(s['E_total'] for s in result['valid_solutions'])
        print(f"  Best E_total: {best_e:.4f}")
        if OPTIMAL_ENERGY is not None:
            print(f"  Gap from optimal: {best_e - OPTIMAL_ENERGY:.4f}")
    print()

In [ ]:
print("="*70)
print("EXTENDED RUN")
print("="*70)

# Find best config by valid rate, then by best energy
best_config = max(all_exp_results, key=lambda x: (x['valid_rate'],
                  -min([s['E_total'] for s in x['valid_solutions']] or [float('inf')])))
lam = best_config['lambda']
print(f"Best config: λ={lam}")

linear, quadratic, offset = qubos[lam]
extended_result = run_sa_experiment(
    linear, quadratic, offset,
    num_reads=200, num_sweeps=30000,
    beta_range=(1.0, 100.0),
    sequence=SEQUENCE, adj=adj_matrix,
    L1=lam[0], L2=lam[1], L3=lam[2],
    seed=123
)

print(f"\nExtended run (200 reads, 30000 sweeps):")
print(f"  Time: {extended_result['solve_time']:.2f}s")
print(f"  Valid: {extended_result['valid_count']}/200 ({extended_result['valid_rate']:.1f}%)")

if extended_result['valid_solutions']:
    energies = [s['E_total'] for s in extended_result['valid_solutions']]
    print(f"\n  E_total Statistics (valid solutions):")
    print(f"    Best (most negative): {min(energies):.4f}")
    print(f"    Mean: {np.mean(energies):.4f}")
    print(f"    Std:  {np.std(energies):.4f}")
    if OPTIMAL_ENERGY is not None:
        print(f"    Gap from optimal: {min(energies) - OPTIMAL_ENERGY:.4f}")

part2_results = {
    'label': '24-residue', 'variables': 576,
    'paths': len(all_paths) if enumeration_complete else None,
    'exact': OPTIMAL_ENERGY if enumeration_complete else None,
    'valid_rate': extended_result['valid_rate'],
    'best': min((s['E_total'] for s in extended_result['valid_solutions']), default=None),
    'reads': 200, 'sweeps': 30000, 'seed': 123,
}


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Valid rate by configuration
ax1 = axes[0]
labels = [f"λ={r['lambda']}\n{r['num_sweeps']//1000}k" for r in all_exp_results]
rates = [r['valid_rate'] for r in all_exp_results]
colors = ['green' if r > 0 else 'red' for r in rates]
ax1.bar(range(len(all_exp_results)), rates, color=colors, alpha=0.7)
ax1.set_xticks(range(len(all_exp_results)))
ax1.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax1.set_ylabel('Valid Solution Rate (%)')
ax1.set_title('24-Residue: Valid Rate by Configuration')

# Energy distribution from extended run
ax2 = axes[1]
if extended_result['valid_solutions']:
    energies = [s['E_total'] for s in extended_result['valid_solutions']]
    ax2.hist(energies, bins=15, edgecolor='black', alpha=0.7)
    if OPTIMAL_ENERGY is not None:
        ax2.axvline(OPTIMAL_ENERGY, color='red', linestyle='--', linewidth=2,
                    label=f'Optimal: {OPTIMAL_ENERGY:.2f}')
    ax2.axvline(min(energies), color='green', linestyle='-', linewidth=2,
                label=f'Best found: {min(energies):.2f}')
    ax2.legend()
    ax2.set_xlabel('E_total')
    ax2.set_ylabel('Count')
ax2.set_title('Extended Run: E_total Distribution')

plt.tight_layout()
plt.savefig('24node_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("="*70)
print("SUMMARY")
print("="*70)

summary_data = []
for r in all_exp_results:
    best = min([s['E_total'] for s in r['valid_solutions']]) if r['valid_solutions'] else None
    summary_data.append({
        'λ': str(r['lambda']),
        'Sweeps': r['num_sweeps'],
        'Time (s)': f"{r['solve_time']:.1f}",
        'Valid %': f"{r['valid_rate']:.0f}",
        'Best E_total': f"{best:.3f}" if best else 'N/A',
    })

df = pd.DataFrame(summary_data)
print(df.to_string(index=False))

if OPTIMAL_ENERGY is not None:
    print(f"\nReference optimal E_total = {OPTIMAL_ENERGY:.4f}")
    print("(Most negative = best)")


---
---

# PART 3: 32-Residue Problem

---
---


# Protein Folding QUBO: 32-Residue Problem

- **32-residue chain** on an **8×4 2D lattice** (32 positions)
- **1,024 binary variables** (32 × 32)
- **Supplied 20-type signed contact matrix; not the cited MJ Table VI; provenance is unknown**

**Ground Truth** (from exhaustive DFS enumeration):
- **Total Hamiltonian paths**: 77,968
- **E_min = -14.25** (exact)
- **Non-sequential contacts at optimum**: 21

**Terminology**:
- **Valid**: All constraints satisfied (E₁ = E₂ = E₃ = 0)
- **Optimal**: Valid solution with lowest (most negative) E_total

In [ ]:
from importlib.util import find_spec
missing = [name for name in ('numpy', 'matplotlib', 'pandas', 'dimod', 'neal')
           if find_spec(name) is None]
if missing:
    raise ImportError(f"Missing {missing}. Install the repository requirements.txt before running this notebook.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
import time
import pandas as pd

from dimod import BinaryQuadraticModel
from neal import SimulatedAnnealingSampler

np.random.seed(42)
print("✓ Imports successful")

## Supplied Contact Matrix

**Convention**: Negative values are favorable. These coefficients do not match the cited paper’s Table VI.

In [ ]:
AMINO_ACIDS = ['A', 'R', 'N', 'D', 'C', 'Q', 'E', 'G', 'H', 'I',
               'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V']

AA_TO_IDX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}

MJ_MATRIX = np.array([
    [-0.20, -0.34, -0.44, -0.42, -0.62, -0.40, -0.42, -0.29, -0.51, -0.73, -0.65, -0.36, -0.64, -0.69, -0.31, -0.34, -0.40, -0.61, -0.53, -0.68],
    [-0.34, -0.15, -0.44, -0.90, -0.55, -0.50, -0.93, -0.40, -0.33, -0.42, -0.45, -0.35, -0.34, -0.41, -0.24, -0.37, -0.32, -0.26, -0.29, -0.37],
    [-0.44, -0.44, -0.28, -0.46, -0.58, -0.44, -0.51, -0.46, -0.54, -0.51, -0.47, -0.46, -0.41, -0.47, -0.34, -0.48, -0.50, -0.36, -0.42, -0.46],
    [-0.42, -0.90, -0.46, -0.19, -0.52, -0.53, -0.49, -0.44, -0.49, -0.42, -0.35, -0.62, -0.33, -0.36, -0.31, -0.48, -0.44, -0.23, -0.30, -0.36],
    [-0.62, -0.55, -0.58, -0.52, -1.54, -0.65, -0.55, -0.54, -0.73, -0.92, -0.88, -0.47, -0.92, -1.01, -0.49, -0.56, -0.61, -0.84, -0.79, -0.91],
    [-0.40, -0.50, -0.44, -0.53, -0.65, -0.29, -0.55, -0.40, -0.49, -0.56, -0.51, -0.46, -0.47, -0.54, -0.31, -0.43, -0.41, -0.39, -0.42, -0.51],
    [-0.42, -0.93, -0.51, -0.49, -0.55, -0.55, -0.22, -0.43, -0.42, -0.44, -0.37, -0.64, -0.35, -0.40, -0.30, -0.47, -0.43, -0.21, -0.27, -0.38],
    [-0.29, -0.40, -0.46, -0.44, -0.54, -0.40, -0.43, -0.14, -0.46, -0.52, -0.46, -0.41, -0.47, -0.51, -0.26, -0.37, -0.39, -0.44, -0.44, -0.47],
    [-0.51, -0.33, -0.54, -0.49, -0.73, -0.49, -0.42, -0.46, -0.34, -0.66, -0.62, -0.35, -0.64, -0.72, -0.37, -0.46, -0.47, -0.62, -0.57, -0.60],
    [-0.73, -0.42, -0.51, -0.42, -0.92, -0.56, -0.44, -0.52, -0.66, -1.00, -0.95, -0.41, -0.96, -1.04, -0.49, -0.52, -0.59, -0.90, -0.83, -0.98],
    [-0.65, -0.45, -0.47, -0.35, -0.88, -0.51, -0.37, -0.46, -0.62, -0.95, -0.91, -0.37, -0.93, -1.01, -0.46, -0.47, -0.53, -0.89, -0.80, -0.93],
    [-0.36, -0.35, -0.46, -0.62, -0.47, -0.46, -0.64, -0.41, -0.35, -0.41, -0.37, -0.18, -0.32, -0.39, -0.23, -0.39, -0.36, -0.19, -0.24, -0.35],
    [-0.64, -0.34, -0.41, -0.33, -0.92, -0.47, -0.35, -0.47, -0.64, -0.96, -0.93, -0.32, -0.89, -0.98, -0.45, -0.47, -0.52, -0.86, -0.78, -0.92],
    [-0.69, -0.41, -0.47, -0.36, -1.01, -0.54, -0.40, -0.51, -0.72, -1.04, -1.01, -0.39, -0.98, -1.10, -0.50, -0.51, -0.57, -0.98, -0.89, -1.02],
    [-0.31, -0.24, -0.34, -0.31, -0.49, -0.31, -0.30, -0.26, -0.37, -0.49, -0.46, -0.23, -0.45, -0.50, -0.16, -0.30, -0.32, -0.42, -0.38, -0.45],
    [-0.34, -0.37, -0.48, -0.48, -0.56, -0.43, -0.47, -0.37, -0.46, -0.52, -0.47, -0.39, -0.47, -0.51, -0.30, -0.33, -0.41, -0.41, -0.41, -0.48],
    [-0.40, -0.32, -0.50, -0.44, -0.61, -0.41, -0.43, -0.39, -0.47, -0.59, -0.53, -0.36, -0.52, -0.57, -0.32, -0.41, -0.38, -0.45, -0.45, -0.55],
    [-0.61, -0.26, -0.36, -0.23, -0.84, -0.39, -0.21, -0.44, -0.62, -0.90, -0.89, -0.19, -0.86, -0.98, -0.42, -0.41, -0.45, -0.86, -0.76, -0.84],
    [-0.53, -0.29, -0.42, -0.30, -0.79, -0.42, -0.27, -0.44, -0.57, -0.83, -0.80, -0.24, -0.78, -0.89, -0.38, -0.41, -0.45, -0.76, -0.69, -0.78],
    [-0.68, -0.37, -0.46, -0.36, -0.91, -0.51, -0.38, -0.47, -0.60, -0.98, -0.93, -0.35, -0.92, -1.02, -0.45, -0.48, -0.55, -0.84, -0.78, -0.96],
])

print(f"MJ Matrix: {MJ_MATRIX.shape}")
print(f"Energy range: [{MJ_MATRIX.min():.2f}, {MJ_MATRIX.max():.2f}]")

## Lattice and Sequence Setup

In [ ]:
def build_2d_lattice(rows, cols):
    """Build adjacency matrix for 2D rectangular lattice."""
    n = rows * cols
    coords = [(x, y) for y in range(cols) for x in range(rows)]
    adj = np.zeros((n, n), dtype=int)
    for i, (x1, y1) in enumerate(coords):
        for j, (x2, y2) in enumerate(coords):
            if abs(x1-x2) + abs(y1-y2) == 1:
                adj[i, j] = 1
    return adj, coords

LATTICE_ROWS, LATTICE_COLS = 8, 4
adj_matrix, lattice_coords = build_2d_lattice(LATTICE_ROWS, LATTICE_COLS)
NUM_POSITIONS = len(lattice_coords)

print(f"Lattice: {LATTICE_ROWS}×{LATTICE_COLS} = {NUM_POSITIONS} positions")

In [ ]:
# First 32 residues of Sequence 5 from Irbäck et al. (2025)
FULL_SEQ_5 = 'ENDVTMDMDPSPCLFRIHNLPRAHSFDRFGWHQFDKYHYKWKWAWAPS'
SEQUENCE = FULL_SEQ_5[:32]

# Ground truth from exhaustive DFS enumeration
E_MIN_EXACT = -14.25
TOTAL_PATHS = 77968
OPTIMAL_CONTACTS = 21

print(f"Sequence: {SEQUENCE}")
print(f"Length: {len(SEQUENCE)}")
print(f"Variables: {len(SEQUENCE) * NUM_POSITIONS}")
print(f"\nGround Truth (DFS enumeration):")
print(f"  Total Hamiltonian paths: {TOTAL_PATHS:,}")
print(f"  Exact E_min: {E_MIN_EXACT}")
print(f"  Contacts at optimum: {OPTIMAL_CONTACTS}")

## Energy Calculation Functions

**E_total = E_MJ + λ₁·E₁ + λ₂·E₂ + λ₃·E₃**

In [ ]:
def compute_E_MJ(chain, b, adj, verbose=False):
    """
    Compute interaction energy.
    MJ values are already negative for favorable, so no negation needed.
    """
    N = len(chain)
    M = adj.shape[0]
    E = 0.0

    for i in range(N):
        for j in range(i+2, N):  # |i-j| > 1
            a_i, a_j = chain[i], chain[j]
            c_ij = MJ_MATRIX[AA_TO_IDX[a_i], AA_TO_IDX[a_j]]
            for n in range(M):
                for m in range(M):
                    if b[i, n] == 1 and b[j, m] == 1 and adj[n, m] == 1:
                        E += c_ij
                        if verbose:
                            print(f"  Contact: {i}({a_i})-{j}({a_j}) at {n}-{m}, C={c_ij}")

    return E


def compute_E1(b):
    """Each residue occupies exactly one site."""
    result = np.sum((np.sum(b, axis=1) - 1) ** 2)
    return int(result)


def compute_E2(b):
    """Each site has at most one residue."""
    M = b.shape[1]
    pair_count = 0
    for n in range(M):
        site_occupancy = np.sum(b[:, n])
        pair_count += site_occupancy * (site_occupancy - 1)
    return int(0.5 * pair_count)


def compute_E3(b, adj):
    """Chain connectivity."""
    N = b.shape[0]
    M = adj.shape[0]
    non_adj = 1 - adj - np.eye(M)
    E3 = 0.0
    for i in range(N - 1):
        E3 += b[i, :] @ non_adj @ b[i+1, :]
    return int(E3)


def total_energy(chain, b, adj, L1=1.0, L2=1.0, L3=1.0, verbose=False):
    """Compute E_total = E_MJ + L1*E1 + L2*E2 + L3*E3"""
    E_MJ = compute_E_MJ(chain, b, adj, verbose)
    E_1 = compute_E1(b)
    E_2 = compute_E2(b)
    E_3 = compute_E3(b, adj)
    total = E_MJ + L1*E_1 + L2*E_2 + L3*E_3

    if verbose:
        print(f"E_MJ={E_MJ:.4f}, E1={E_1}, E2={E_2}, E3={E_3}, E_total={total:.4f}")

    return total, (E_MJ, E_1, E_2, E_3)


def is_valid(breakdown):
    """Valid = all constraints satisfied."""
    _, E1, E2, E3 = breakdown
    return E1 == 0 and E2 == 0 and E3 == 0


print("✓ Energy functions defined")

## QUBO Construction

In [ ]:
def build_protein_qubo(sequence, adj, lambda1=1.5, lambda2=2.0, lambda3=2.0):
    """Build QUBO. Minimizing finds most negative E_total."""
    N = len(sequence)
    M = adj.shape[0]

    linear = defaultdict(float)
    quadratic = defaultdict(float)
    offset = 0.0

    def bit_idx(residue, position):
        return residue * M + position

    # E_MJ: Add C_ij directly
    for i in range(N):
        for j in range(i + 2, N):
            C_ij = MJ_MATRIX[AA_TO_IDX[sequence[i]], AA_TO_IDX[sequence[j]]]
            for n in range(M):
                for m in range(M):
                    if adj[n, m] == 1:
                        b_i, b_j = bit_idx(i, n), bit_idx(j, m)
                        key = (min(b_i, b_j), max(b_i, b_j))
                        quadratic[key] += C_ij

    # E1
    for i in range(N):
        for n in range(M):
            linear[bit_idx(i, n)] += lambda1 * (-1)
        for n in range(M):
            for m in range(n + 1, M):
                key = (bit_idx(i, n), bit_idx(i, m))
                quadratic[key] += lambda1 * 2
        offset += lambda1

    # E2
    for n in range(M):
        for i in range(N):
            for j in range(i + 1, N):
                key = (bit_idx(i, n), bit_idx(j, n))
                quadratic[key] += lambda2

    # E3
    non_adj = 1 - adj - np.eye(M)
    for i in range(N - 1):
        for n in range(M):
            for m in range(M):
                if non_adj[n, m] == 1:
                    b_i, b_j = bit_idx(i, n), bit_idx(i + 1, m)
                    key = (min(b_i, b_j), max(b_i, b_j))
                    quadratic[key] += lambda3

    return dict(linear), dict(quadratic), offset


lambda_configs = [(1.5, 2.0, 2.0), (2.0, 2.0, 2.0), (3.0, 4.0, 4.0)]

print(f"Building QUBOs ({len(SEQUENCE)} × {NUM_POSITIONS} = {len(SEQUENCE) * NUM_POSITIONS} variables)...")
qubos = {}
for lam in lambda_configs:
    start = time.time()
    linear, quadratic, offset = build_protein_qubo(SEQUENCE, adj_matrix, *lam)
    qubos[lam] = (linear, quadratic, offset)
    print(f"  λ={lam}: {len(quadratic)} quadratic terms ({time.time()-start:.1f}s)")

## Solution Validation

In [ ]:
def validate_solution(sample, sequence, adj, L1, L2, L3):
    """Validate solution and compute E_total."""
    N = len(sequence)
    M = adj.shape[0]

    if set(sample) != set(range(N * M)) or any(value not in (0, 1) for value in sample.values()):
        raise ValueError('sample must contain exactly N*M binary variables')

    b = np.zeros((N, M))
    for bit_idx, val in sample.items():
        if val == 1:
            i, n = bit_idx // M, bit_idx % M
            if i < N:
                b[i, n] = 1

    E_total, breakdown = total_energy(sequence, b, adj, L1, L2, L3)
    valid = is_valid(breakdown)

    return valid, E_total, breakdown


print("✓ Validation function defined")

## Simulated Annealing Experiments

In [ ]:
def run_sa_experiment(linear, quadratic, offset, num_reads, num_sweeps, beta_range,
                      sequence, adj, L1, L2, L3, seed=None):
    """Run SA and return statistics."""
    bqm = BinaryQuadraticModel(linear, quadratic, offset, vartype='BINARY')
    sampler = SimulatedAnnealingSampler()

    start = time.time()
    result = sampler.sample(bqm, num_reads=num_reads, num_sweeps=num_sweeps,
                            beta_range=beta_range, seed=seed)
    solve_time = time.time() - start

    valid_solutions = []
    all_results = []

    for datum in result.data():
        valid, E_total, breakdown = validate_solution(datum.sample, sequence, adj, L1, L2, L3)
        if not np.isclose(datum.energy, E_total, rtol=0, atol=1e-6):
            raise AssertionError('Sampler QUBO energy disagrees with direct evaluation')
        all_results.append({'valid': valid, 'E_total': E_total,
                            'QUBO_energy': datum.energy, 'breakdown': breakdown})
        if valid:
            valid_solutions.append({'E_total': E_total, 'QUBO_energy': datum.energy})

    return {
        'solve_time': solve_time,
        'num_reads': num_reads,
        'num_sweeps': num_sweeps,
        'valid_count': len(valid_solutions),
        'valid_solutions': valid_solutions,
        'valid_rate': len(valid_solutions) / num_reads * 100,
        'all_results': all_results
    }

In [ ]:
print("="*70)
print(f"SIMULATED ANNEALING EXPERIMENTS")
print(f"32-residue problem, Exact E_min = {E_MIN_EXACT}")
print("="*70)
print()

all_exp_results = []

test_configs = [
    ((1.5, 2.0, 2.0), 5000, 50, (1.0, 50.0)),
    ((2.0, 2.0, 2.0), 5000, 50, (1.0, 50.0)),
    ((3.0, 4.0, 4.0), 5000, 50, (1.0, 50.0)),
    ((2.0, 2.0, 2.0), 10000, 50, (1.0, 50.0)),
    ((2.0, 2.0, 2.0), 20000, 50, (1.0, 100.0)),
]

for lam, sweeps, reads, beta in test_configs:
    print(f"λ={lam}, sweeps={sweeps}")

    linear, quadratic, offset = qubos[lam]
    result = run_sa_experiment(linear, quadratic, offset, reads, sweeps, beta,
                               SEQUENCE, adj_matrix, lam[0], lam[1], lam[2], seed=42)
    result['lambda'] = lam
    all_exp_results.append(result)

    print(f"  Time: {result['solve_time']:.1f}s")
    print(f"  Valid: {result['valid_count']}/{reads} ({result['valid_rate']:.1f}%)")

    if result['valid_solutions']:
        best_e = min(s['E_total'] for s in result['valid_solutions'])
        gap = best_e - E_MIN_EXACT
        print(f"  Best E_total: {best_e:.2f} (optimal: {E_MIN_EXACT}, gap: {gap:+.2f})")
    else:
        avg_e1 = np.mean([r['breakdown'][1] for r in result['all_results']])
        print(f"  Avg E1 violations: {avg_e1:.1f}")
    print()

In [ ]:
print("="*70)
print("EXTENDED RUN")
print("="*70)

lam = (2.0, 2.0, 2.0)
linear, quadratic, offset = qubos[lam]

print(f"\nRunning extended SA (100 reads, 50000 sweeps)...")
extended_result = run_sa_experiment(
    linear, quadratic, offset,
    num_reads=100, num_sweeps=50000,
    beta_range=(1.0, 100.0),
    sequence=SEQUENCE, adj=adj_matrix,
    L1=lam[0], L2=lam[1], L3=lam[2],
    seed=123
)

print(f"\nResults:")
print(f"  Solve time: {extended_result['solve_time']:.1f}s")
print(f"  Valid: {extended_result['valid_count']}/100 ({extended_result['valid_rate']:.1f}%)")

if extended_result['valid_solutions']:
    energies = [s['E_total'] for s in extended_result['valid_solutions']]
    best_e = min(energies)
    print(f"\n  E_total Statistics (valid solutions):")
    print(f"    Best (most negative): {best_e:.4f}")
    print(f"    Mean: {np.mean(energies):.4f}")
    print(f"    Std:  {np.std(energies):.4f}")
    print(f"\n  Comparison to exact optimum:")
    print(f"    Exact E_min: {E_MIN_EXACT}")
    print(f"    Gap: {best_e - E_MIN_EXACT:+.4f}")
    if abs(best_e - E_MIN_EXACT) < 0.01:
        print(f"    *** OPTIMAL FOUND! ***")
else:
    avg_e1 = np.mean([r['breakdown'][1] for r in extended_result['all_results']])
    print(f"\n  No valid solutions. Avg E1 violations: {avg_e1:.1f}")

part3_results = {
    'label': '32-residue', 'variables': 1024, 'paths': TOTAL_PATHS,
    'exact': E_MIN_EXACT, 'valid_rate': extended_result['valid_rate'],
    'best': min((s['E_total'] for s in extended_result['valid_solutions']), default=None),
    'reads': 100, 'sweeps': 50000, 'seed': 123,
}


In [ ]:
if extended_result['valid_solutions']:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Energy distribution
    energies = [s['E_total'] for s in extended_result['valid_solutions']]
    axes[0].hist(energies, bins=20, edgecolor='black', alpha=0.7)
    axes[0].axvline(E_MIN_EXACT, color='r', linestyle='--', linewidth=2, label=f'Exact optimum ({E_MIN_EXACT})')
    axes[0].axvline(min(energies), color='g', linestyle='--', linewidth=2, label=f'Best found ({min(energies):.2f})')
    axes[0].set_xlabel('E_total')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Energy Distribution (Valid Solutions)')
    axes[0].legend()

    # Valid rate across experiments
    labels = [f"λ={r['lambda']}\n{r['num_sweeps']}sw" for r in all_exp_results]
    valid_rates = [r['valid_rate'] for r in all_exp_results]
    colors = ['green' if r > 50 else 'orange' if r > 20 else 'red' for r in valid_rates]
    axes[1].bar(range(len(labels)), valid_rates, color=colors, edgecolor='black')
    axes[1].set_xticks(range(len(labels)))
    axes[1].set_xticklabels(labels, fontsize=8)
    axes[1].set_ylabel('Valid Rate (%)')
    axes[1].set_title('Valid Rate by Configuration')
    axes[1].set_ylim(0, 100)

    plt.tight_layout()
    plt.show()
else:
    print("No valid solutions to visualize.")

## Summary

In [ ]:
print("="*70)
print("32-NODE PROTEIN FOLDING SUMMARY")
print("="*70)

our_best = min([s['E_total'] for s in extended_result['valid_solutions']]) if extended_result['valid_solutions'] else None
our_valid_rate = extended_result['valid_rate']
our_time = extended_result['solve_time']

print(f"\nProblem Size:")
print(f"  Lattice: {LATTICE_ROWS}×{LATTICE_COLS} 2D")
print(f"  Residues: {len(SEQUENCE)}")
print(f"  Variables: {len(SEQUENCE) * NUM_POSITIONS}")
print(f"  Hamiltonian paths: {TOTAL_PATHS:,}")

print(f"\nGround Truth (DFS enumeration):")
print(f"  Exact E_min: {E_MIN_EXACT}")
print(f"  Optimal contacts: {OPTIMAL_CONTACTS}")

print(f"\nSimulated Annealing Results:")
print(f"  Solve time: {our_time:.1f}s")
print(f"  Valid rate: {our_valid_rate:.1f}%")
if our_best:
    print(f"  Best E_total: {our_best:.4f}")
    gap = our_best - E_MIN_EXACT
    print(f"  Gap from optimal: {gap:+.4f}")
    if abs(gap) < 0.01:
        print(f"  Status: *** OPTIMAL FOUND ***")
    elif gap < 0.5:
        print(f"  Status: Near-optimal")
    else:
        print(f"  Status: Suboptimal")

print("\n" + "="*70)
print("VERIFICATION STATUS")
print("="*70)
print("""
This 32-residue problem is an exact benchmark for the supplied 2D contact model:

1. Ground truth established via exhaustive DFS enumeration
   - All 77,968 Hamiltonian paths enumerated
   - Exact minimum energy: -14.25

2. Same supplied contact matrix as the 8- and 24-residue problems
   - 24-node (4×6 2D): E_min = -10.05 ✓ verified
   - 32-node (8×4 2D): E_min = -14.25 ✓ verified

3. QUBO formulation validated
   - Constraints correctly encoded
   - Energy calculation matches direct computation
""")

In [ ]:
# Figures use results produced by this notebook run.
from pathlib import Path
import matplotlib.pyplot as plt

summary = [part1_results, part2_results, part3_results]
headers = ['Problem', 'Variables', 'Valid paths', 'Exact min', 'SA best', 'Valid %', 'SA reads', 'Sweeps']
rows = []
for item in summary:
    rows.append([
        item['label'], str(item['variables']),
        f"{item['paths']:,}" if item['paths'] is not None else 'incomplete',
        f"{item['exact']:.2f}" if item['exact'] is not None else 'unknown',
        f"{item['best']:.2f}" if item['best'] is not None else 'none',
        f"{item['valid_rate']:.1f}", str(item['reads']), str(item['sweeps'])
    ])

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.axis('off')
table = ax.table(cellText=rows, colLabels=headers, cellLoc='center', loc='center')
table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1, 1.7)
ax.set_title('Protein folding QUBO results (current notebook run)')
fig.tight_layout()
fig.savefig('figure1_results_table.png', dpi=150, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
labels = [item['label'] for item in summary]
axes[0].bar(labels, [item['valid_rate'] for item in summary])
axes[0].set_ylabel('Valid samples (%)')
axes[0].set_title('Feasibility by problem size')
gaps = [item['best'] - item['exact'] if item['best'] is not None and item['exact'] is not None else float('nan') for item in summary]
axes[1].bar(labels, gaps)
axes[1].set_ylabel('Best valid energy − exact minimum')
axes[1].set_title('Best observed energy gap')
fig.tight_layout()
fig.savefig('figure2_scaling_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('Figures reflect these runs; sweep schedules and read counts differ by problem size.')


---
# Conclusions

The exact path counts and minimum energies above refer to these specific 2D lattices and sequences. SA results are stochastic and depend on the sampler, seed, penalty weights, and sweep schedule. Compare the measured valid rate and energy gap in the generated figure; neither a valid sample nor a low QUBO energy alone proves that a run found the physical optimum.

The 48-residue results in the cited paper use a different 3D lattice and should not be treated as measurements from this notebook.
